In [2]:
import os
import re
import pandas as pd
import json
import glob
import csv
import ast
import numpy as np
import pandas as pd
import openai
import random
import re
from flatten_json import flatten
from time import process_time
import ipywidgets as widgets
from IPython.display import display, clear_output
from retry import retry
from zhon.hanzi import punctuation
import pycorrector as pc
import pinyin_tone_converter as ptc
import warnings
warnings.filterwarnings("ignore")

In [3]:
import math
import pykakasi
import re
import string
from zhon.hanzi import punctuation
import random


In [142]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from scipy.linalg import norm
 
def tfidf_similarity(s1, s2):
    def add_space(s):
        return ' '.join(s)
     
    # Add space in between-words
    s1, s2 = add_space(s1), add_space(s2)
    # convert to TF Vector
    cv = TfidfVectorizer(tokenizer=lambda s: s.split())
    corpus = [s1, s2]
    vectors = cv.fit_transform(corpus).toarray()
    # Count TF Coefficent
    return np.dot(vectors[0], vectors[1]) / (norm(vectors[0]) * norm(vectors[1]))

In [143]:
from simhash import Simhash
def simhash_demo(text_a, text_b):
    """
    Get two texts' similarity
    :param text_a:
    :param text_b:
    :return:
    """
    a_simhash = Simhash(text_a)
    b_simhash = Simhash(text_b)
    max_hashbit = max(len(bin(a_simhash.value)), len(bin(b_simhash.value)))
    # Hamming Distance
    distince = a_simhash.distance(b_simhash)
    print(distince)
    similar = 1 - distince / max_hashbit
    return similar

In [19]:
df=pd.read_csv("Japanese_correction_datasets/jwtd/gen/jwtd_test_combined.csv")
df.head()

,page,pre_rev,post_rev,pre_loss,post_loss,post_text,diffs,pre_txt,post_txt,category,...,gpt35_romanji_gen,gpt35_incorrectstrings_gen,gpt35_romanji_time,gpt35_incorrectstrings_time,opennmt_model_textsim,gpt_35_romanji_textsim,gpt35_incorrectstrings_textsim,opennmt_model_simhash_textsim,gpt35_romanji_simhash_textsim,gpt35_incorrectstrings_simhash_textsim
0,881,5958209,9736989,251.51,245.60,1987年に局部銀河群（大マゼラン雲）内において超新星爆発(SN1987A)が起こったため、...,"[{'pre': 'を', 'post': 'の'}]",1987年に局部銀河群（大マゼラン雲）内において超新星爆発(SN1987A)が起こったため、...,1987年に局部銀河群（大マゼラン雲）内において超新星爆発(SN1987A)が起こったため、...,substitution,...,0,1987年に局部銀河群（大マゼラン雲）内において超新星爆発(SN1987A)が起こったため、...,0,0.066108,1.0,0,0.983143,1.0,0,0.875000
1,887,31045825,31065369,189.78,175.93,制約が多く、独占的・長期に連載されるため（作者の死によって連載終了となることも多い）マンネリ...,"[{'pre': 'か', 'post': 'も'}]",制約が多く、独占的・長期に連載されるため（作者の死によって連載終了となることも多い）マンネリ...,制約が多く、独占的・長期に連載されるため（作者の死によって連載終了となることも多い）マンネリ...,substitution,...,0,制約が多く、独占的・長期に連載されるため（作者の死によって連載終了となることも多い）マンネリ...,0,0.011402,1.0,0,0.982616,1.0,0,0.876923
2,1253,38176846,38227527,201.24,189.63,一説にはグロンギに敗れた他の部族が奴隷として無理やりグロンギに改造された傘下に加えられた集団...,"[{'pre': 'れれ', 'post': 'られ'}]",一説にはグロンギに敗れた他の部族が奴隷として無理やりグロンギに改造された傘下に加えれれた集団...,一説にはグロンギに敗れた他の部族が奴隷として無理やりグロンギに改造された傘下に加えれれた集団...,substitution,...,0,一説にはグロンギに敗れた他の部族が奴隷として無理やりグロンギに改造された傘下に加えられた集団...,0,0.011292,1.0,0,0.983214,1.0,0,0.892308
3,1253,35848754,35899590,98.14,94.01,フロント部に金色のプレートなど、随所に金色の装甲が発生している。,"[{'pre': 'に', 'post': 'の'}]",フロント部に金色のプレートなど、随所に金色に装甲が発生している。,フロント部に金色のプレートなど、随所に金色に装甲が発生している。,substitution,...,0,フロント部に金色のプレートなど、随所に金色の装甲が発生している。,0,0.009064,1.0,0,0.975900,1.0,0,0.772727
4,1432,67727119,68197779,528.34,521.10,このような議論を通じ、少なくとも老子なる人物が生きたであろう時代と『老子道徳経』が作られた時...,"[{'pre': 'もと', 'post': 'もの'}]",このような議論を通じ、少なくとも老子なる人物が生きたであろう時代と『老子道徳経』が作られた時...,このような議論を通じ、少なくとも老子なる人物が生きたであろう時代と『老子道徳経』が作られた時...,substitution,...,0,このような議論を通じ、少なくとも老子なる人物が生きたであろう時代と『老子道徳経』が作られた時...,0,0.013184,1.0,0,0.997275,1.0,0,0.909091


In [121]:
import pykakasi

kks = pykakasi.kakasi()

def df_kana(df):
    
    df=df.assign(pre_romanji=0,post_romanji=0)
    

    for i in range(df.shape[0]):
        
        result = kks.convert(df.pre_txt.iloc[i])
        result1 = kks.convert(df.post_txt.iloc[i])
        lst=[]
        lst1=[]
        for item in result:
            lst.append(item['hepburn'])
        df.pre_romanji.iloc[i]=' '.join(lst)
        
        for item in result1:
            lst1.append(item['hepburn'])
        df.post_romanji.iloc[i]=' '.join(lst1)
            
            #print("{}: kana '{}', hiragana '{}', romaji: '{}'".format(item['orig'], item['kana'], item['hira'], item['hepburn']))

    return df

In [122]:
df=df_kana(df)

In [123]:
df.tail()

,page,pre_rev,post_rev,pre_loss,post_loss,post_txt,diffs,pre_txt,category,pre_romanji,...,gpt35_incorrectstrings_time,opennmt_model_textsim,gpt_35_romanji_textsim,gpt35_incorrectstrings_textsim,opennmt_model_simhash_textsim,gpt35_romanji_simhash_textsim,gpt35_incorrectstrings_simhash_textsim,pesedo_romanji,noise_num,pesedo_details
8537,3877588,71062937,71163561,213.38,206.69,ビザンツ帝国の支配は820年代の後半までに終わり、クレタ島はアンダルス（イベリア半島）からや...,"[{'pre': '広範', 'post': '後半'}]",ビザンツ帝国の支配は820年代の広範までに終わり、クレタ島はアンダルス（イベリア半島）からや...,kanji-conversion,bizantsu teikoku no shihai ha 820 nendai no ko...,...,0.0,0.966541,0,0.0,0.924242,0,0.0,bizantsu teikoku no shihai ha 820 nendai no ko...,0,[]
8538,3932005,72407958,72458531,44.27,46.42,乗り物や人混みが苦手。,"[{'pre': '人込み', 'post': '人混み'}]",乗り物や人込みが苦手。,kanji-conversion,norimono ya hitogomi ga nigate.,...,0.0,0.820021,0,0.0,0.742424,0,0.0,norimono ya hitogomi ga nigate.,1,"[{'origin_token': 'o', 'noise_token': '', 'noi..."
8539,3932005,72407958,72458531,58.89,62.67,れんげと同様人混みが苦手。,"[{'pre': '人込み', 'post': '人混み'}]",れんげと同様人込みが苦手。,kanji-conversion,rengeto douyou hitogomi ga nigate.,...,0.0,0.847762,0,0.0,0.742424,0,0.0,rengeto douyou hitogomi ga nigate.,0,[]
8540,3937616,72517512,72517518,476.97,476.58,権翼はなおも「陛下は小信を重んじ、社稷を軽んじられております。臣が見ます所、一度行かせれば二...,"[{'pre': '生かせれ', 'post': '行かせれ'}]",権翼はなおも「陛下は小信を重んじ、社稷を軽んじられております。臣が見ます所、一度生かせれば二...,kanji-conversion,kenyoku hanaomo( heika ha shou shin wo omon ji...,...,0.0,0.991141,0,0.0,0.878788,0,0.0,kenyoku hanaomo( heika ha shou shin wo omon ji...,2,"[{'origin_token': 'u', 'noise_token': 'u', 'no..."
8541,3947988,72805433,72807446,439.61,438.45,フォーブスの「アメリカで最も裕福なセレブリティ」（フォーブスのアメリカでもっともゆうふくなセ...,"[{'pre': '史', 'post': '誌'}]",フォーブスの「アメリカで最も裕福なセレブリティ」（フォーブスのアメリカでもっともゆうふくなセ...,kanji-conversion,foobusu no( amerika de mottomo yuufuku na sere...,...,0.0,0.994050,0,0.0,0.878788,0,0.0,foobusu no( amerika de mottomo yuufuku na sere...,0,[]


In [139]:
import math
import re
import numpy as np
import pandas as pd
import string
from zhon.hanzi import punctuation
import random




t26_keyboard = {'q': (0, 0), 'w': (0, 1), 'e': (0, 2), 'r': (0, 3), 't': (0, 4), 'y': (0, 5), 'u': (0, 6),
                    'i': (0, 7), 'o': (0, 8), 'p': (0, 9),
                    'a': (1, 0), 's': (1, 1), 'd': (1, 2), 'f': (1, 3), 'g': (1, 4), 'h': (1, 5), 'j': (1, 6),
                    'k': (1, 7), 'l': (1, 8),
                    'z': (2, 0), 'x': (2, 1), 'c': (2, 2), 'v': (2, 3), 'b': (2, 4), 'n': (2, 5), 'm': (2, 6)
                    }
t9_keyboard = {
      'a': (0, 1), 'b': (0, 1), 'c': (0, 1),
      'd': (0, 2), 'e': (0, 2), 'f': (0, 2),
      'g': (1, 0), 'h': (1, 0), 'i': (1, 0),
      'j': (1, 1), 'k': (1, 1), 'l': (1, 1),
      'm': (1, 2), 'n': (1, 2), 'o': (1, 2),
      'p': (2, 0), 'q': (2, 0), 'r': (2, 0), 's': (2, 0),
      't': (2, 1), 'u': (2, 1), 'v': (2, 1),
      'w': (2, 2), 'x': (2, 2), 'y': (2, 2), 'z': (2, 2)
  }




# Define a function to calculate the Euclidean distance between two points
def calculate_distance(point1, point2):
    x1, y1 = point1
    x2, y2 = point2
    return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

coordinates = list(t26_keyboard.values())



def fixed_strings(fstr): 
    E=ast.literal_eval(fstr)
    filtered_chars = []
    for item in E:
        pre_value = item['pre']
        if len(pre_value) >= 2:
            filtered_chars.append(pre_value[:2])
        else:
            filtered_chars.append(pre_value)
        
#filtered_chars=list(set(filtered_chars))
    kks = pykakasi.kakasi()
    s=[]
    for it in filtered_chars:
        result = kks.convert(it)
        for item in result:
            s.append(item['hepburn'])
    
    return s



def typo_romanji(fstr): 
    E=ast.literal_eval(fstr)
    filtered_chars1 = []
    filtered_chars2 = []
    for item in E:
        pre_value = item['pre']
        post_value = item['post']
        if len(pre_value) >= 2:
            filtered_chars1.append(pre_value[:2])
        else:
            filtered_chars1.append(pre_value)
            
        if len(post_value) >= 2:
            filtered_chars2.append(post_value[:2])
        else:
            filtered_chars2.append(post_value)
        
    filtered_chars1=list(set(filtered_chars1))
    filtered_chars2=list(set(filtered_chars2))
    kks = pykakasi.kakasi()
    pre=[]
    post=[]
    for it in filtered_chars1:
        result = kks.convert(it)
        for item in result:
            pre.append(item['hepburn'])
    
    for it in filtered_chars2:
        result = kks.convert(it)
        for item in result:
            post.append(item['hepburn'])
            
            
    return pre,post






punctuations = punctuation + string.punctuation + '1234567890・★ →'

def kanji_pesedo_build(valid_kanji,fixed_lst):

    valid_kanji_new=valid_kanji.lower()
    valid_kanji_new=' '.join(filter(lambda x: x not in punctuations, valid_kanji_new)).split()
    valid_kanji_new=[item for item in valid_kanji_new if item not in fixed_lst]

    #if random.random() <= 0.9:
        #error_num=random.randint(1, 2)
    #else:
        #error_num=0

    error_num=random.randint(0,2)
    noise_token=random.choices(valid_kanji_new,k=error_num)

    invalid_kanji=valid_kanji.split()
    valid_position=[]
    details=[]

    for item in noise_token:
        initials=' '.join(item).split()
        finals=initials
        kanji=str(item)

        error_type = random.choice(['missing', 'extra','typo'])
        if error_type == 'extra':
            position = random.randint(0, len(kanji))
            error = random.choice(initials + finals)
            kanji = kanji[:position] + error + kanji[position:]


        else:
            if error_type == 'missing':
                position = random.randint(0, len(kanji) - 1)
                kanji = kanji[:position] + kanji[position + 1:]

            else:
                p=random.random()
                if p < 0.6:
                    position = random.randint(0, len(kanji)-1)
                    reference_point = t26_keyboard[kanji[position]]
                    # Filter coordinates based on distance <= 1 from the reference point
                    filtered_coordinates = [coord for coord in coordinates if calculate_distance(reference_point, coord) == 1]
                    filtered_lst=[]
                    for ig in filtered_coordinates:
                        filtered_dict = [key for key, value in t26_keyboard.items() if value == ig ]
                        filtered_lst.append(filtered_dict[0])
                        noise=random.choice(filtered_lst)
                        kanji=list(kanji)
                        kanji[position]=noise
                        kanji=' '.join(kanji)
                else:
                    position = random.randint(0, len(kanji) - 1)
                    kanji=list(kanji)
                    kanji[position],kanji[position-1]=kanji[position-1],kanji[position]
                    kanji=' '.join(kanji)


        details.append({'origin_token': item, 'noise_token': kanji, 'noise_type':error_type})

        for index, value in enumerate(invalid_kanji): 
            if value == item:
                valid_position.append(index)
                invalid_kanji[index]=kanji
    invalid_kanji=' '.join(invalid_kanji)

    
    return {'invalid_kanji_gen':invalid_kanji, 'noise_num':error_num,'details':details }


if __name__ == '__main__':

    df=df.assign(pesedo_romanji=0, noise_num=0,pesedo_details=0)
    for i in range(df.shape[0]):
        if df.iloc[i].category!="kanji-conversion":
            df.pesedo_romanji.iloc[i]==df.iloc[i].pre_romanji
            details=[]
            details.append({'origin_token': ''.join(typo_romanji(df.diffs.iloc[i])[0]), 'noise_token': ''.join(typo_romanji(df.diffs.iloc[i])[1]), 'noise_type':df.category.iloc[i]})
            df.pesedo_details.iloc[i]=details
            df.noise_num.iloc[i]=df.iloc[i].pesedo_details.count("noise_token")

        else:    
            valid_kanji=df.pre_romanji.iloc[i]
            fixed_lst=fixed_strings(df.iloc[i].diffs)

            t=kanji_pesedo_build(valid_kanji,fixed_lst)
            df.pesedo_romanji.iloc[i]=t.get("invalid_kanji_gen")
            df.noise_num.iloc[i]=t.get("noise_num")
            df.pesedo_details.iloc[i]=t.get("details")

In [140]:
df

,page,pre_rev,post_rev,pre_loss,post_loss,post_txt,diffs,pre_txt,category,pre_romanji,...,gpt35_incorrectstrings_time,opennmt_model_textsim,gpt_35_romanji_textsim,gpt35_incorrectstrings_textsim,opennmt_model_simhash_textsim,gpt35_romanji_simhash_textsim,gpt35_incorrectstrings_simhash_textsim,pesedo_romanji,noise_num,pesedo_details
0,881,5958209,9736989,251.51,245.60,1987年に局部銀河群（大マゼラン雲）内において超新星爆発(SN1987A)が起こったため、...,"[{'pre': 'を', 'post': 'の'}]",1987年に局部銀河群（大マゼラン雲）内において超新星爆発(SN1987A)が起こったため、...,substitution,1987 nen ni kyokubu ginga gun ( dai mazeran ku...,...,0.066108,1.000000,0,0.983143,1.000000,0,0.875000,0,0,"[{'origin_token': 'wo', 'noise_token': 'no', '..."
1,887,31045825,31065369,189.78,175.93,制約が多く、独占的・長期に連載されるため（作者の死によって連載終了となることも多い）マンネリ...,"[{'pre': 'か', 'post': 'も'}]",制約が多く、独占的・長期に連載されるため（作者の死によって連載終了となることも多い）マンネリ...,substitution,"seiyaku ga ooku, dokusenteki ・ chouki ni rensa...",...,0.011402,1.000000,0,0.982616,1.000000,0,0.876923,0,0,"[{'origin_token': 'ka', 'noise_token': 'mo', '..."
2,1253,38176846,38227527,201.24,189.63,一説にはグロンギに敗れた他の部族が奴隷として無理やりグロンギに改造された傘下に加えられた集団...,"[{'pre': 'れれ', 'post': 'られ'}]",一説にはグロンギに敗れた他の部族が奴隷として無理やりグロンギに改造された傘下に加えれれた集団...,substitution,issetsu niha gurongi ni yabure ta hokano buzok...,...,0.011292,1.000000,0,0.983214,1.000000,0,0.892308,0,0,"[{'origin_token': 'rere', 'noise_token': 'rare..."
3,1253,35848754,35899590,98.14,94.01,フロント部に金色のプレートなど、随所に金色の装甲が発生している。,"[{'pre': 'に', 'post': 'の'}]",フロント部に金色のプレートなど、随所に金色に装甲が発生している。,substitution,"furonto bu ni kin'iro no pureeto nado, zuisho ...",...,0.009064,1.000000,0,0.975900,1.000000,0,0.772727,0,0,"[{'origin_token': 'ni', 'noise_token': 'no', '..."
4,1432,67727119,68197779,528.34,521.10,このような議論を通じ、少なくとも老子なる人物が生きたであろう時代と『老子道徳経』が作られた時...,"[{'pre': 'もと', 'post': 'もの'}]",このような議論を通じ、少なくとも老子なる人物が生きたであろう時代と『老子道徳経』が作られた時...,substitution,"konoyouna giron wo tsuuji, sukunakutomo roushi...",...,0.013184,1.000000,0,0.997275,1.000000,0,0.909091,0,0,"[{'origin_token': 'moto', 'noise_token': 'mono..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8537,3877588,71062937,71163561,213.38,206.69,ビザンツ帝国の支配は820年代の後半までに終わり、クレタ島はアンダルス（イベリア半島）からや...,"[{'pre': '広範', 'post': '後半'}]",ビザンツ帝国の支配は820年代の広範までに終わり、クレタ島はアンダルス（イベリア半島）からや...,kanji-conversion,bizantsu teikoku no shihai ha 820 nendai no ko...,...,0.000000,0.966541,0,0.000000,0.924242,0,0.000000,bizantsu teikoku no shihai ha 820 nendai no ko...,2,"[{'origin_token': 'i', 'noise_token': 'ii', 'n..."
8538,3932005,72407958,72458531,44.27,46.42,乗り物や人混みが苦手。,"[{'pre': '人込み', 'post': '人混み'}]",乗り物や人込みが苦手。,kanji-conversion,norimono ya hitogomi ga nigate.,...,0.000000,0.820021,0,0.000000,0.742424,0,0.000000,norimono ya hitogomi ga nigate.,0,[]
8539,3932005,72407958,72458531,58.89,62.67,れんげと同様人混みが苦手。,"[{'pre': '人込み', 'post': '人混み'}]",れんげと同様人込みが苦手。,kanji-conversion,rengeto douyou hitogomi ga nigate.,...,0.000000,0.847762,0,0.000000,0.742424,0,0.000000,rengeto douyou hitogomi ga nigate.,2,"[{'origin_token': 'o', 'noise_token': 'p', 'no..."
8540,3937616,72517512,72517518,476.97,476.58,権翼はなおも「陛下は小信を重んじ、社稷を軽んじられております。臣が見ます所、一度行かせれば二...,"[{'pre': '生かせれ', 'post': '行かせれ'}]",権翼はなおも「陛下は小信を重んじ、社稷を軽んじられております。臣が見ます所、一度生かせれば二...,kanji-conversion,kenyoku hanaomo( heika ha shou shin wo omon ji...,...,0.000000,0.991141,0,0.000000,0.878788,0,0.000000,kenyoku hanaomo( heika ha shou shin wo omon ji...,2,"[{'origin_token': 'o', 'noise_token': 'oo', 'n..."


In [144]:
df.columns

Index(['page', 'pre_rev', 'post_rev', 'pre_loss', 'post_loss', 'post_txt',
       'diffs', 'pre_txt', 'category', 'pre_romanji', 'post_romanji',
       'opennmt_model_gen', 'gpt35_romanji_gen', 'gpt35_incorrectstrings_gen',
       'gpt35_romanji_time', 'gpt35_incorrectstrings_time',
       'opennmt_model_textsim', 'gpt_35_romanji_textsim',
       'gpt35_incorrectstrings_textsim', 'opennmt_model_simhash_textsim',
       'gpt35_romanji_simhash_textsim',
       'gpt35_incorrectstrings_simhash_textsim', 'pesedo_romanji', 'noise_num',
       'pesedo_details'],
      dtype='object')

In [145]:
# Test

print(df.loc[2].pre_txt)
#print(df.loc[2].post_txt)
print(df.loc[2].post_txt)

df.loc[4].pre_romanji

df.loc[4].post_romanji

df.loc[4].diffs

df.loc[4].pesedo_details

,page,pre_rev,post_rev,pre_loss,post_loss,post_txt,diffs,pre_txt,category,pre_romanji,...,gpt35_incorrectstrings_time,opennmt_model_textsim,gpt_35_romanji_textsim,gpt35_incorrectstrings_textsim,opennmt_model_simhash_textsim,gpt35_romanji_simhash_textsim,gpt35_incorrectstrings_simhash_textsim,pesedo_romanji,noise_num,pesedo_details
2998,985570,18433991,18448517,191.50,183.11,「チューチューチュー」ならスーパーリーチ確定で、猫が登場してネズミを捕まえれば確変大当たり確定。,"[{'pre': 'が', 'post': ''}]",「チューチューチュー」ならスーパーリーチ確定で、猫が登場してがネズミを捕まえれば確変大当たり確定。,insertion,"( chuuchuuchuu )nara suupaariichi kakutei de, ...",...,0.0,0.995183,0,0.0,0.939394,0,0.0,0,0,"[{'origin_token': 'ga', 'noise_token': '', 'no..."
2999,985570,15304875,15304907,235.16,225.16,また、突入時の回転でリーチが掛かった場合は大当たり確定となり、突入時の回転で青以外の泥棒や大...,"[{'pre': 'と', 'post': ''}]",また、突入時の回転でリーチが掛かった場合はと大当たり確定となり、突入時の回転で青以外の泥棒や...,insertion,"mata, totsunyuu tokino kaiten de riichi ga kak...",...,0.0,0.996214,0,0.0,0.893939,0,0.0,0,0,"[{'origin_token': 'to', 'noise_token': '', 'no..."
3000,991318,67204287,67216756,232.02,223.92,土谷は両親、弟妹、生徒の親らの前で合田から「オウムをやめると言わなければ殺すど」、母親から「...,"[{'pre': 'た', 'post': ''}]",た土谷は両親、弟妹、生徒の親らの前で合田から「オウムをやめると言わなければ殺すど」、母親から...,insertion,"ta tsuchiya ha ryoushin, teimai, seito no oya ...",...,0.0,0.994104,0,0.0,0.892308,0,0.0,0,0,"[{'origin_token': 'ta', 'noise_token': '', 'no..."
3001,991318,66430056,66444069,149.48,133.93,あくまで「尊師の意思に従う。わたしから喋るわけにはいかない」というスタンスを貫いた。,"[{'pre': 'た', 'post': ''}]",あくまで「尊師の意思に従う。わたしから喋るわけにはいかない」というスタンスを貫いたた。,insertion,akumade( sonshi no ishi ni shitagau. watashika...,...,0.0,0.992994,0,0.0,0.921875,0,0.0,0,0,"[{'origin_token': 'ta', 'noise_token': '', 'no..."
3002,991318,66226316,66226392,257.99,248.97,村井秀夫や遠藤誠一の指示には非科学的なものも多く、二人の指示を聞いて失敗するべきなのかと麻原...,"[{'pre': 'の', 'post': ''}]",村井秀夫や遠藤誠一のの指示には非科学的なものも多く、二人の指示を聞いて失敗するべきなのかと麻...,insertion,murai hideo ya endou seiichi nono shiji niha h...,...,0.0,0.996855,0,0.0,0.969231,0,0.0,0,0,"[{'origin_token': 'no', 'noise_token': '', 'no..."
3003,991318,70158721,70158982,358.48,347.53,「実の兄のような気持ちを持っていた」などという大ウソをつきながら青山さんを検察・警察に売り渡...,"[{'pre': 'は', 'post': ''}]",「実の兄のような気持ちを持っていた」などという大ウソをつきながら青山さんをは検察・警察に売り...,insertion,( mi no ani noyouna kimochi wo motsu teita)nad...,...,0.0,0.998200,0,0.0,0.924242,0,0.0,0,0,"[{'origin_token': 'ha', 'noise_token': '', 'no..."
3004,991318,66489417,66495121,166.10,155.00,9月に入り、水戸支部長の岐部哲也から出家を促されるが家族と縁を絶てずに断った。,"[{'pre': 'い', 'post': ''}]",9月に入り、水戸支部長の岐部哲也から出家を促されるが家族と縁を絶てずにい断った。,insertion,"9 gatsu ni iri, mito shibuchou no kibe tetsuya...",...,0.0,0.978749,0,0.0,0.892308,0,0.0,0,0,"[{'origin_token': 'i', 'noise_token': '', 'noi..."
3005,991318,66409391,66409870,357.82,348.57,地下鉄サリン事件直前の3月20日未明、大阪道場に警察の強制捜査が入ったことを知り、クシティガ...,"[{'pre': 'で', 'post': ''}]",地下鉄サリン事件直前の3月20日未明、大阪道場に警察の強制捜査が入ったことを知り、クシティガ...,insertion,chikatetsu sarin jiken chokuzen no 3 gatsu 20 ...,...,0.0,0.996639,0,0.0,0.909091,0,0.0,0,0,"[{'origin_token': 'de', 'noise_token': '', 'no..."


In [147]:
dff=df.copy()

In [148]:
def textsim_opennmt_corrector(df):
    
    #for i in range(df.shape[0]):
        #if type(df.opennmt_model_gen.iloc[i]) == int:
            #df.opennmt_model_gen.iloc[i]=  str(df.opennmt_model_gen.iloc[i])


    
    for i in range(df.shape[0]):
        gen_text=''.join(filter(lambda x: x not in punctuation, df.opennmt_model_gen.iloc[i])) 
        org_text=''.join(filter(lambda x: x not in punctuation, df.post_txt.iloc[i]))
        if gen_text == org_text:
            df.opennmt_model_textsim.iloc[i]=1
            df.opennmt_model_simhash_textsim.iloc[i]=1

        else:
            
            df.opennmt_model_simhash_textsim.iloc[i]=simhash_demo(gen_text,org_text)
            
            df.opennmt_model_textsim.iloc[i]=tfidf_similarity(gen_text,org_text)
    
    return df

In [149]:
df=textsim_opennmt_corrector(df)
df.head()

8
7
7
15
6
12
7
6
6
10
12
14
12
6
10
7
7
4
4
13
10
12
4
10
9
14
7
3
4
9
1
3
19
17
7
5
3
10
1
7
5
19
13
11
8
6
6
3
3
7
6
7
7
5
6
5
11
26
7
13
10
15
5
4
8
9
9
10
3
6
16
3
13
5
7
7
9
17
8
5
8
11
8
7
13
11
7
16
11
9
3
8
15
7
6
8
3
6
5
13
6
13
4
8
10
7
10
4
12
2
4
11
6
6
0
6
5
6
17
13
7
3
9
3
3
4
5
7
4
18
10
9
4
9
9
10
8
6
6
16
2
14
10
4
14
6
8
15
13
5
8
5
9
9
7
5
6
12
6
6
17
7
1
8
9
20
8
5
20
8
7
12
4
5
10
8
1
3
8
6
7
10
6
7
6
4
8
7
10
5
17
6
10
16
2
15
5
5
6
11
4
12
5
9
5
7
8
16
9
11
5
3
6
11
6
2
4
3
10
9
8
6
8
6
5
10
6
3
9
9
13
15
8
7
7
3
10
7
5
4
3
10
9
6
8
6
8
7
7
6
8
5
3
3
7
9
4
6
11
2
13
6
4
9
11
13
18
11
6
9
4
14
13
11
10
6
10
5
10
13
7
7
4
8
6
6
5
13
5
5
4
14
5
5
9
15
5
9
2
6
8
6
3
8
5
6
11
6
17
13
11
8
3
9
7
19
11
7
6
21
4
9
10
6
6
4
6
6
15
3
15
9
10
15
5
16
2
8
9
5
5
7
7
5
5
5
13
11
6
5
9
10
11
5
4
10
26
4
8
8
8
10
12
12
9
4
5
8
4
7
5
11
21
10
7
10
8
2
8
15
13
4
6
5
5
3
9
9
5
7
11
6
7
6
6
5
7
12
4
15
2
13
16
4
6
11
4
13
11
10
9
6
9
14
9
13
9
5
8
9
8
14
8
9
10
9
13
7
8
15
15
2
10


6
7
13
7
6
8
4
7
11
1
11
5
6
17
13
4
10
10
5
6
12
13
3
12
16
8
8
7
4
8
6
9
10
9
13
10
7
12
3
5
4
16
17
11
5
8
4
9
7
2
2
5
8
5
6
2
8
9
10
5
8
9
4
12
9
9
6
5
3
16
9
6
7
2
7
7
10
8
6
4
2
7
4
12
5
9
5
9
5
4
5
11
6
5
4
10
5
5
4
8
20
12
15
9
7
10
5
5
10
6
5
7
8
4
8
7
7
11
6
4
7
4
5
9
9
2
11
12
11
2
6
6
5
3
4
4
12
13
5
3
8
4
7
7
2
11
12
12
5
5
11
7
2
10
14
7
6
8
7
6
11
6
7
4
21
6
4
9
1
8
3
9
14
10
18
7
5
5
11
4
5
16
7
1
4
11
16
2
5
5
7
9
18
8
8
8
13
5
3
8
8
7
9
13
8
5
11
8
9
6
11
7
6
9
11
9
5
3
11
7
4
3
28
4
9
11
7
9
5
8
5
7
3
16
4
13
10
4
7
8
9
8
11
13
13
7
1
11
5
7
3
6
29
6
5
10
3
9
6
4
9
10
3
6
9
12
2
4
12
7
4
10
9
8
5
9
5
10
9
14
5
4
4
5
16
2
11
7
8
4
8
12
13
25
20
3
10
10
9
9
8
10
2
5
16
4
11
7
4
8
5
11
18
5
11
11
8
10
17
8
7
6
5
4
4
13
5
5
7
9
18
17
22
7
12
6
9
7
4
6
4
9
14
6
8
6
5
9
3
9
11
6
7
8
10
8
3
9
4
13
4
6
3
7
10
9
2
11
6
2
7
6
6
7
7
8
10
26
8
11
9
7
4
6
10
8
4
11
12
5
12
2
4
2
4
7
12
8
12
15
4
10
14
10
9
8
5
6
10
6
7
10
6
8
9
4
3
6
5
8
7
7
5
9
8
13
11
12
8
10
8
7
10
9
6
9
8
10


9
11
3
6
6
4
17
8
7
10
5
8
6
5
10
16
4
10
13
8
11
6
5
16
8
10
6
10
13
6
10
15
16
8
2
10
13
16
7
13
16
6
4
13
15
12
7
8
11
12
15
6
13
12
8
9
17
4
4
4
12
8
11
11
11
7
11
10
7
0
11
12
5
16
23
18
9
8
9
10
10
20
8
10
7
7
9
12
6
8
11
5
5
7
7
4
5
6
7
5
10
6
4
9
3
12
5
5
9
9
6
6
6
10
8
6
10
9
7
9
18
6
17
5
6
9
9
3
12
2
8
4
9
7
9
7
13
6
9
6
5
16
5
10
7
9
5
3
13
18
11
11
10
13
9
6
11
12
6
11
18
4
8
8
6
6
12
11
6
3
7
4
7
9
7
18
12
10
7
6
7
8
8
17
12
5
11
15
3
7
6
11
10
5
12
8
9
8
9
6
10
8
12
3
10
19
8
12
9
10
9
6
10
5
7
10
8
4
16
7
10
9
10
7
6
6
15
9
12
1
3
4
6
9
4
6
7
4
7
17
7
8
8
11
9
4
5
7
5
8
6
3
8
10
11
9
8
16
6
14
3
18
9
7
12
11
8
19
15
11
4
8
9
17
7
12
12
6
8
10
6
10
2
8
11
5
27
5
11
14
5
6
7
4
11
8
11
11
3
3
12
9
14
7
9
12
9
11
13
5
13
8
4
10
7
4
7
6
6
4
16
12
6
9
8
4
7
11
4
9
7
12
9
10
9
11
9
4
12
5
11
11
11
5
11
20
16
7
12
7
4
10
25
7
6
16
7
12
9
5
9
3
5
16
3
9
7
4
13
3
16
4
9
3
9
9
9
8
12
7
12
13
15
7
9
7
5
8
5
3
11
3
6
11
7
4
15
15
8
16
3
11
4
10
11
8
15
10
12
5
12
4
7
8
8
8
13
12
8
1

,page,pre_rev,post_rev,pre_loss,post_loss,post_txt,diffs,pre_txt,category,pre_romanji,...,gpt35_incorrectstrings_time,opennmt_model_textsim,gpt_35_romanji_textsim,gpt35_incorrectstrings_textsim,opennmt_model_simhash_textsim,gpt35_romanji_simhash_textsim,gpt35_incorrectstrings_simhash_textsim,pesedo_romanji,noise_num,pesedo_details
0,881,5958209,9736989,251.51,245.60,1987年に局部銀河群（大マゼラン雲）内において超新星爆発(SN1987A)が起こったため、...,"[{'pre': 'を', 'post': 'の'}]",1987年に局部銀河群（大マゼラン雲）内において超新星爆発(SN1987A)が起こったため、...,substitution,1987 nen ni kyokubu ginga gun ( dai mazeran ku...,...,0.066108,0.983143,0,0.983143,0.875000,0,0.875000,0,0,"[{'origin_token': 'wo', 'noise_token': 'no', '..."
1,887,31045825,31065369,189.78,175.93,制約が多く、独占的・長期に連載されるため（作者の死によって連載終了となることも多い）マンネリ...,"[{'pre': 'か', 'post': 'も'}]",制約が多く、独占的・長期に連載されるため（作者の死によって連載終了となることも多い）マンネリ...,substitution,"seiyaku ga ooku, dokusenteki ・ chouki ni rensa...",...,0.011402,0.982616,0,0.982616,0.892308,0,0.876923,0,0,"[{'origin_token': 'ka', 'noise_token': 'mo', '..."
2,1253,38176846,38227527,201.24,189.63,一説にはグロンギに敗れた他の部族が奴隷として無理やりグロンギに改造された傘下に加えられた集団...,"[{'pre': 'れれ', 'post': 'られ'}]",一説にはグロンギに敗れた他の部族が奴隷として無理やりグロンギに改造された傘下に加えれれた集団...,substitution,issetsu niha gurongi ni yabure ta hokano buzok...,...,0.011292,0.983214,0,0.983214,0.892308,0,0.892308,0,0,"[{'origin_token': 'rere', 'noise_token': 'rare..."
3,1253,35848754,35899590,98.14,94.01,フロント部に金色のプレートなど、随所に金色の装甲が発生している。,"[{'pre': 'に', 'post': 'の'}]",フロント部に金色のプレートなど、随所に金色に装甲が発生している。,substitution,"furonto bu ni kin'iro no pureeto nado, zuisho ...",...,0.009064,0.975900,0,0.975900,0.772727,0,0.772727,0,0,"[{'origin_token': 'ni', 'noise_token': 'no', '..."
4,1432,67727119,68197779,528.34,521.10,このような議論を通じ、少なくとも老子なる人物が生きたであろう時代と『老子道徳経』が作られた時...,"[{'pre': 'もと', 'post': 'もの'}]",このような議論を通じ、少なくとも老子なる人物が生きたであろう時代と『老子道徳経』が作られた時...,substitution,"konoyouna giron wo tsuuji, sukunakutomo roushi...",...,0.013184,0.997275,0,0.997275,0.909091,0,0.909091,0,0,"[{'origin_token': 'moto', 'noise_token': 'mono..."


In [172]:
def textsim_gpt35_pycorrector(df):
    for i in range(df.shape[0]):
        if df.gpt35_incorrectstrings_gen.iloc[i]!="0":
            if type(df.gpt35_incorrectstrings_gen.iloc[i]) == int:
                df.gpt35_incorrectstrings_gen.iloc[i]=  str(df.gpt35_incorrectstrings_gen.iloc[i])
       
            gen_text=''.join(filter(lambda x: x not in punctuation, df.gpt35_incorrectstrings_gen.iloc[i])) 
            org_text=''.join(filter(lambda x: x not in punctuation, df.post_txt.iloc[i]))
            if gen_text == org_text:
                df.gpt35_incorrectstrings_simhash_textsim.iloc[i]=1
                df.gpt35_incorrectstrings_textsim.iloc[i]=1

            else:

                df.gpt35_incorrectstrings_simhash_textsim.iloc[i]=simhash_demo(gen_text,org_text)

                df.gpt35_incorrectstrings_textsim.iloc[i]=tfidf_similarity(gen_text,org_text)

    return df

In [173]:
df=textsim_gpt35_pycorrector(df)
df.head()

11
12
15
0
19
15
4
4
0
10
9
12
7
14
15
6
2
21
18
11
12
10
8
12
7
11
6
7
7
9
10
11
9
13
16
13
11
15
4
9
6
8
9
13
8
7
6
15
8
13
15
6
4
13
13
11
12
6
0
0
8
0
12
4
16
10
13
12
4
11
6
6
2
13
6
13
7
3
6
6
0
9
6
0
7
12
12
13
13
13
13
11
14
6
8
5
10
5
5
6
7
1
8
6
7
0
8
20
13
12
4
4
9
4
14
0
12
8
6
8
5
17
6
6
8
1
4
19
5
5
9
11
10
11
0
5
0
10
4
8
9
8
7
10
4
9
13
15
7
11
10
4
7
9
5
8
8
8
10
0
0
3
1
0
6
11
4
10
0
9
11
12
9
4
10
8
16
11
10
6
14
5
10
13
7
7
8
11
0
9
0
0
14
17
9
11
12
5
0
10
11
0
9
6
17
11
10
6
6
15
3
15
17
15
5
16
1
6
5
15
5
11
4
9
31
4
8
12
9
4
7
6
7
0
2
15
13
5
14
5
7
8
0
6
8
10
6
0
9
17
0
9
11
14
9
9
17
11
7
14
0
8
0
12
4
1
0
16
12
13
13
7
11
11
9
6
5
10
13
6
6
9
15
11
5
17
4
7
7
10
6
11
6
16
8
9
10
9
14
10
11
2
6
5
0
9
9
17
13
7
10
8
5
10
18
15
4
19
0
12
0
6
4
11
12
9
16
13
11
6
9
5
0
0
10
8
5
6
6
8
15
2
7
5
11
8
12
10
6
8
7
0
13
5
11
11
10
8
5
14
5
9
10
7
11
7
8
9
0
12
11
10
0
10
8
7
10
8
0
12
14
6
8
9
0
10
0
9
0
15
8
3
10
7
11
15
8
15
14
4
15
11
14
2
6
0
12
6
9
5
20
13
4
6
0
0

,page,pre_rev,post_rev,pre_loss,post_loss,post_txt,diffs,pre_txt,category,pre_romanji,...,gpt35_incorrectstrings_time,opennmt_model_textsim,gpt_35_romanji_textsim,gpt35_incorrectstrings_textsim,opennmt_model_simhash_textsim,gpt35_romanji_simhash_textsim,gpt35_incorrectstrings_simhash_textsim,pesedo_romanji,noise_num,pesedo_details
0,881,5958209,9736989,251.51,245.60,1987年に局部銀河群（大マゼラン雲）内において超新星爆発(SN1987A)が起こったため、...,"[{'pre': 'を', 'post': 'の'}]",1987年に局部銀河群（大マゼラン雲）内において超新星爆発(SN1987A)が起こったため、...,substitution,1987 nen ni kyokubu ginga gun ( dai mazeran ku...,...,0.066108,0.983143,0,1.000000,0.875000,0,1.000000,0,0,"[{'origin_token': 'wo', 'noise_token': 'no', '..."
1,887,31045825,31065369,189.78,175.93,制約が多く、独占的・長期に連載されるため（作者の死によって連載終了となることも多い）マンネリ...,"[{'pre': 'か', 'post': 'も'}]",制約が多く、独占的・長期に連載されるため（作者の死によって連載終了となることも多い）マンネリ...,substitution,"seiyaku ga ooku, dokusenteki ・ chouki ni rensa...",...,0.011402,0.982616,0,0.988372,0.892308,0,0.830769,0,0,"[{'origin_token': 'ka', 'noise_token': 'mo', '..."
2,1253,38176846,38227527,201.24,189.63,一説にはグロンギに敗れた他の部族が奴隷として無理やりグロンギに改造された傘下に加えられた集団...,"[{'pre': 'れれ', 'post': 'られ'}]",一説にはグロンギに敗れた他の部族が奴隷として無理やりグロンギに改造された傘下に加えれれた集団...,substitution,issetsu niha gurongi ni yabure ta hokano buzok...,...,0.011292,0.983214,0,1.000000,0.892308,0,1.000000,0,0,"[{'origin_token': 'rere', 'noise_token': 'rare..."
3,1253,35848754,35899590,98.14,94.01,フロント部に金色のプレートなど、随所に金色の装甲が発生している。,"[{'pre': 'に', 'post': 'の'}]",フロント部に金色のプレートなど、随所に金色に装甲が発生している。,substitution,"furonto bu ni kin'iro no pureeto nado, zuisho ...",...,0.009064,0.975900,0,1.000000,0.772727,0,1.000000,0,0,"[{'origin_token': 'ni', 'noise_token': 'no', '..."
4,1432,67727119,68197779,528.34,521.10,このような議論を通じ、少なくとも老子なる人物が生きたであろう時代と『老子道徳経』が作られた時...,"[{'pre': 'もと', 'post': 'もの'}]",このような議論を通じ、少なくとも老子なる人物が生きたであろう時代と『老子道徳経』が作られた時...,substitution,"konoyouna giron wo tsuuji, sukunakutomo roushi...",...,0.013184,0.997275,0,1.000000,0.909091,0,1.000000,0,0,"[{'origin_token': 'moto', 'noise_token': 'mono..."


In [141]:
dff=df.copy()

In [174]:
df.to_csv("Japanese_correction_datasets/jwtd/gen/jwtd_test_combined.csv",index=False)

In [178]:
df.category.value_counts()

kanji-conversion    3061
insertion           2127
substitution        1689
deletion            1665
Name: category, dtype: int64